In [11]:
users = [
    {"id": 1, "total": 100, "coupon": "P20"},
    {"id": 2, "total": 150, "coupon": "P30"},
    {"id": 3, "total": 200, "coupon": "P40"}
]

discount = {
    "P20": (0.2, 0),
    "P20": (0.3, 0),
    "P20": (0, 10)
}

In [21]:
for user in users:
    print(f"keys: {user['id']} values: {user['total']}")

keys: 1 values: 100
keys: 2 values: 150
keys: 3 values: 200


In [26]:
list = ["iced tea", "hot tea"]
unique = [chai for chai in list if "iced" in chai]
print(unique)

['iced tea']


In [1]:
list = [1,2,3,4,5]
[x for x in list if x%2 == 0]

[2, 4]

In [20]:
import asyncio

async def task(name):
    print(f"{name} started")
    time.sleep(2)
    print(f"{name} finished")



In [21]:
import asyncio as ass

async def task2(name):
    print(f"{name} started")

    await ass.sleep(2)

    print(f"{name} finished")

name = input("Enter a task name: ")

await task2(name)



 started
 finished


In [22]:
import asyncio

async def main():
    print("Hello")

await main()

Hello


In [23]:
async def main():
    await asyncio.gather(
        task("A"),
        task("B")
    )

await main()

A started
A finished
B started
B finished


In [25]:
import asyncio

async def get_data(number):
    await asyncio.sleep(2)
    return number * 2

async def main():
    results = await asyncio.gather(
        get_data(1),
        get_data(2),
        get_data(3)
    )

    print(results)

await main()

[2, 4, 6]


In [4]:
#data structure
from pydantic import BaseModel, Field
class User(BaseModel):
    id: int
    name: str
    age: int = Field(gt=0, le=100)

In [5]:
#using User
user = User(id="1", name="ali", age=5)
print(user.name)

ali


In [6]:
from typing import List, Dict
class Cart(BaseModel):
    userId: int
    items: List[str]
    quantities: Dict[str, int]

In [7]:
data = {
    "userId": 1,
    "items": ["laptop", "Iphone", "Playstation"],
    "quantities": {"laptop": 1, "Iphone": 2, "Playstation": 5},
}

In [8]:
cart = Cart(**data)
cart.userId

1

In [9]:
cart.items

['laptop', 'Iphone', 'Playstation']

In [10]:
cart.quantities["laptop"]

1

In [12]:
class Employee(BaseModel):
    id: int
    name: str = Field(..., min_length=3, max_length=50, description="Employee name", example="John Doe")
    salary: float = Field(..., gt=0, description="Employee salary", example=50000.0)


/var/folders/kj/wcfxsq8d1kbf3n6vc9dnjcwr0000gn/T/ipykernel_10299/96571202.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  name: str = Field(..., min_length=3, max_length=50, description="Employee name", example="John Doe")


In [13]:
from pydantic import BaseModel, model_validator

class User(BaseModel):
    password: str
    confirm_password: str

    @model_validator(mode="after")
    def passwords_match(self):
        if self.password != self.confirm_password:
            raise ValueError("Passwords don't match")
        return self

In [19]:
user = User(password="secret", confirm_password="secret")
print(user.password)

secret


In [31]:
class Employees(BaseModel):
    name: str

    @model_validator(mode="after")
    def names(self) -> "Employees":
        if self.name == "Umar":
            raise ValueError("Name cannot be Umar")
        return self

employee = Employees(name="Ali")
print(employee.name)

Ali


In [23]:
from pydantic import computed_field

class Area(BaseModel):
    length: float
    width: float

    @computed_field
    @property
    def area(self) -> float:
        return self.length * self.width


In [27]:
area = Area(length=5, width=10)
print(area.length)  # Output: 5
print(area.width)   # Output: 10
print(area.area)    # Output: 50

5.0
10.0
50.0


In [32]:
class Booking(BaseModel):
    user_id: int
    room_id: int
    nights: int = Field(..., gt=0, description="Number of nights for the booking", example=3)
    rate_per_night: float

    @computed_field
    @property
    def total_cost(self) -> float:
        return self.nights * self.rate_per_night

/var/folders/kj/wcfxsq8d1kbf3n6vc9dnjcwr0000gn/T/ipykernel_10299/2934813106.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  nights: int = Field(..., gt=0, description="Number of nights for the booking", example=3)


In [33]:
booking = Booking(user_id=1, room_id=101, nights=3, rate_per_night=100.0)
print(booking.user_id)
print(booking.room_id)
print(booking.nights)
print(booking.rate_per_night)
print(booking.total_cost) #we can calculate a value in property and function behaves like a new attribute value

1
101
3
100.0
300.0


In [36]:
#----------------COMBINING ALL THE ABOVE CONCEPTS-----------------

from pydantic import field_validator


class Registration(BaseModel):
    username: str
    password: str
    confirm_password: str

    @field_validator("username")
    def clean_username(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("Username cannot be empty")
        return value

    @model_validator(mode="after")
    def passwords_match(self) -> "Registration":
        if self.password != self.confirm_password:
            raise ValueError("Passwords do not match")
        return self

    @computed_field
    @property
    def display_name(self) -> str:
        return self.username.title()

user = Registration(
    username=" ali ",
    password="secret123",
    confirm_password="secret123",
)

print(user.display_name)  # Ali
print(user.model_dump())  # Includes display_name

Ali
{'username': 'ali', 'password': 'secret123', 'confirm_password': 'secret123', 'display_name': 'Ali'}


In [38]:
class UserNames(BaseModel):
    first_name: str
    last_name: str

    @field_validator("first_name")
    def capitalize_names(cls, value: str):
        if not value.istitle():
            raise ValueError("First name must be capitalized")
        return value

In [53]:
class Email(BaseModel):
    email: str

    @field_validator("email")
    def validate_email(cls, value: str):
        return value.lower().strip()

In [54]:
email = Email(email="aBc@gmail.com")

In [55]:
email.email

'abc@gmail.com'

In [56]:
email.validate_email(email.email)

'abc@gmail.com'

In [60]:
class Address(BaseModel):
    street: str
    city: str
    zip_code: str

In [61]:
class City(BaseModel):
    name: str
    population: int
    address: Address

    def display_info(self) -> str:
        return f"City: {self.name}, Population: {self.population}, Address: {self.address.street}, {self.address.city}, {self.address.zip_code}"    
    

In [63]:
city = {
    "name": "New York",
    "population": 8419600,
    "address": Address(
        street="123 Main St",
        city="New York",
        zip_code="10001",
    ),
}

city1 = City(**city)
print(city1.display_info())

City: New York, Population: 8419600, Address: 123 Main St, New York, 10001


In [65]:
from typing import Optional

class Comment(BaseModel):
    id: int
    content: str
    replies: Optional[List['Comment']]= None

Comment.model_rebuild()  # Rebuild the model to resolve forward references


In [ ]:
#We read json data from apis like this and we can easily use the data in pydentic

comment_data = {
    "id": 1,
    "content": "This is a comment",
    "replies": [
        Comment(id=2, content="This is a reply"),
        Comment(id=3, content="This is another reply"),
    ],
}

comment = Comment(**comment_data)
print(comment)

id=1 content='This is a comment' replies=[Comment(id=2, content='This is a reply', replies=None), Comment(id=3, content='This is another reply', replies=None)]
